# FedMamba-SALT: Split 1 FedProx Federated Experiment

> **Clean paper notebook for the severe non-IID Split 1 FedProx experiment.**

This notebook runs the paper pipeline:
1. Federated SALT self-supervised pretraining with **FedProx**.
2. Client split inspection and non-IID distribution plots.
3. Training diagnostics for convergence, collapse detection, timing, and memory.
4. Frozen-encoder **linear probe** evaluation.
5. Fully federated fine-tuning of the pre-trained encoder.
6. Test-time augmentation (TTA) evaluation for the federated fine-tuned checkpoint.
7. Paper-ready figure checklist and Google Drive artifact backup.

### Recommended Split 1 Regimen

| Stage | Algorithm | Key settings |
|---|---|---|
| SSL pretraining | FedProx | `mu=0.005`, `E_epoch=1`, `rounds=300`, `lr=5e-4` |
| Linear probe | Central readout only | frozen encoder, supervised classifier only |
| Federated fine-tuning | FedProx + server momentum | `--algo fedavgm --mu 0.01`, `E_epoch=3`, `rounds=80`, `lr=1e-3` |
| TTA evaluation | Post-training inference | best federated fine-tuned checkpoint, `n_tta=8` |

### Required Paper Artifacts
- Client class distribution and augmentation-pair sanity check.
- FedProx training curves: global loss, per-client loss, client variance, embedding std, timing, GPU memory, and dashboard.
- Linear-probe confusion matrix and classification report.
- Federated fine-tuning curves, client diagnostics, confusion matrix, ROC curve, and report.
- TTA summary, TTA comparison plot, and threshold-sweep plot.
- Final summary comparison plot and Drive artifact manifest.

### Architecture
- **Teacher**: Frozen MAE ViT-B/16.
- **Student**: Inception-Mamba encoder (`embed_dim=448`, `depth=6`, `out_dim=768`).
- **Paper alignment**: uses the InceptionMamba dual-branch block (Inception local branch + SSM global branch + channel attention + channel shuffle), customized with `16x16` patches and no patch merging so all 196 tokens align with the ViT-B/16 teacher.
- **Pretraining loss**: SALT centered and standardized dense patch distillation.
- **Fine-tuning head**: attention pooling over dense patch tokens.


## Section 1: Environment Setup

### 1.1 📁 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 📍 Set Paths (⚠️ EDIT THESE)

Configure **four** paths: repo location, dataset root, teacher checkpoint, and the **split** to run.

In [ ]:
# ============================================================
# EDIT THESE PATHS TO MATCH YOUR GOOGLE DRIVE LAYOUT
# ============================================================
DRIVE_REPO          = "/content/drive/MyDrive/fedmamba_salt"
DRIVE_DATASET_DRIVE = "/content/drive/MyDrive/Retina"   # must have train/, test/, labels.csv
DRIVE_CKPT          = "/content/drive/MyDrive/original/retina/split1/retina_pretrain_mae_base_split1_checkpoint-1599.pth"
ZIP_PATH            = "/content/drive/MyDrive/Retina.zip"  # optional local-copy source

# ============================================================
# FEDERATED SPLIT 1 PRETRAINING SETTINGS
# ============================================================
SPLIT_TYPE  = "split_1"
N_CLIENTS   = 5
MAX_ROUNDS  = 300
E_EPOCH     = 1
MU          = 0.005
BATCH_SIZE  = 128
LR          = 5e-4
NUM_CLASSES = 2
NUM_WORKERS = 10

ALGO_NAME = "FedProx" if MU > 0 else "FedAvg"
print(f"Algorithm:  {ALGO_NAME} (mu={MU})")
print(f"Split:      {SPLIT_TYPE}")
print(f"Clients:    {N_CLIENTS}")
print(f"Rounds:     {MAX_ROUNDS}")
print(f"E_epoch:    {E_EPOCH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"LR:         {LR}")


### 1.2.b 📦 Copy Dataset Locally (Fixes Google Drive I/O Bottleneck)

In [ ]:
import os
import time
import zipfile

LOCAL_DATASET = "/content/Retina_local"

start_time = time.time()
if os.path.exists(ZIP_PATH):
    print(f"Preparing local dataset from {ZIP_PATH} -> {LOCAL_DATASET}")
    if not os.path.exists(LOCAL_DATASET):
        os.makedirs(LOCAL_DATASET, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATASET)
    else:
        print(f"Directory {LOCAL_DATASET} already exists. Skipping extraction.")

    candidate = os.path.join(LOCAL_DATASET, "Retina")
    DRIVE_DATASET = candidate if os.path.exists(candidate) else LOCAL_DATASET
elif os.path.exists(DRIVE_DATASET_DRIVE):
    print("Dataset zip not found; using Drive dataset directory directly.")
    DRIVE_DATASET = DRIVE_DATASET_DRIVE
else:
    raise FileNotFoundError(
        f"Neither ZIP_PATH nor DRIVE_DATASET_DRIVE exists:\n"
        f"  ZIP_PATH={ZIP_PATH}\n"
        f"  DRIVE_DATASET_DRIVE={DRIVE_DATASET_DRIVE}"
    )

elapsed = time.time() - start_time
print(f"Dataset ready in {elapsed:.1f} seconds")
print(f"Dataset path for training: {DRIVE_DATASET}")


In [ ]:
import os
import shutil

# If the dataset was copied from a zip that did not include central/test.csv,
# copy central split metadata from the original Drive dataset.
source_central = os.path.join(DRIVE_DATASET_DRIVE, "central")
local_central = os.path.join(DRIVE_DATASET, "central")

if os.path.exists(source_central):
    os.makedirs(local_central, exist_ok=True)
    for item in os.listdir(source_central):
        src = os.path.join(source_central, item)
        dst = os.path.join(local_central, item)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    print(f"Verified central split files in {local_central}")
else:
    print(f"Central split source not found: {source_central}")


In [ ]:
import os

print(f"Verifying checkpoint path: {DRIVE_CKPT}")
if os.path.exists(DRIVE_CKPT):
    print(f"✓ Checkpoint file found at {DRIVE_CKPT}")
else:
    print(f"✗ Checkpoint file NOT found at {DRIVE_CKPT}")
    parent_dir = os.path.dirname(DRIVE_CKPT)
    if os.path.isdir(parent_dir):
        print(f"  Listing contents of parent directory '{parent_dir}':")
        for item in os.listdir(parent_dir):
            print(f"    - {item}")
    else:
        print(f"  Parent directory '{parent_dir}' does not exist.")

print("Please ensure the DRIVE_CKPT path is absolutely correct and matches a file in your Google Drive.")

### 1.3 — Copy Repo to Local Colab Filesystem

Copying to `/content/` avoids slow Google Drive I/O on every Python import. The dataset stays on Drive — DataLoader handles large sequential reads efficiently.

In [ ]:
import shutil, os

LOCAL_REPO = "/content/fedmamba_salt"
if os.path.exists(LOCAL_REPO):
    shutil.rmtree(LOCAL_REPO)
shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)
print(f"Repo copied to {LOCAL_REPO}")

In [ ]:
import shutil, os
# Copy the MAE checkpoint to the expected local path
CKPT_DIR = os.path.join(LOCAL_REPO, "data", "ckpts")
CKPT_PATH = os.path.join(CKPT_DIR, "mae_vit_base.pth")
os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_PATH):
    shutil.copy2(DRIVE_CKPT, CKPT_PATH)
    size_mb = os.path.getsize(CKPT_PATH) / 1e6
    print(f"Checkpoint copied: {size_mb:.1f} MB")
else:
    print(f"Checkpoint already present: {CKPT_PATH}")

# Define output paths for federated experiments
OUTPUT_DIR = os.path.join(LOCAL_REPO, "outputs", f"{ALGO_NAME.lower()}_{SPLIT_TYPE}")
DRIVE_OUTPUT = os.path.join(DRIVE_REPO, "outputs", f"{ALGO_NAME.lower()}_{SPLIT_TYPE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"Local output:  {OUTPUT_DIR}")
print(f"Drive output:  {DRIVE_OUTPUT}")

In [ ]:
import os

print(f"Contents of local output directory ({OUTPUT_DIR}):")
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        print(f"  - {f}")
else:
    print(f"  Directory not found: {OUTPUT_DIR}")

print(f"\nContents of Drive output directory ({DRIVE_OUTPUT}):")
if os.path.exists(DRIVE_OUTPUT):
    for f in os.listdir(DRIVE_OUTPUT):
        print(f"  - {f}")
else:
    print(f"  Directory not found: {DRIVE_OUTPUT}")

print("\nPlease ensure 'ckpt_latest.pth' is present in one of these locations for resuming.")

### 1.4 — Install Dependencies

In [ ]:
%%capture install_output
# Install order matters: causal-conv1d before mamba-ssm.
# --no-build-isolation lets the build see the existing PyTorch/CUDA in Colab.
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

# Teacher backbone and utilities.
!pip install timm==0.3.2
!pip install einops PyYAML matplotlib tqdm pandas seaborn scikit-image scikit-learn


In [ ]:
# Show any errors from install (the output is captured above)
install_text = install_output.stdout + install_output.stderr
errors = [line for line in install_text.split('\n') if 'error' in line.lower() or 'failed' in line.lower()]
if errors:
    print("⚠️  Install errors detected:")
    for e in errors:
        print(f"  {e}")
else:
    print("✓ All dependencies installed successfully.")

### 1.5 — Patch timm for PyTorch 2.x Compatibility

`timm==0.3.2` imports `torch._six` which was removed in PyTorch 2.0+. This shim **must** run before any `import timm`.

> **If you restart the runtime**, re-run from this cell onwards.

In [ ]:
import collections.abc, sys, types

# Create a shim module that satisfies timm's internal import
mock_six = types.ModuleType("torch._six")
mock_six.container_abcs = collections.abc
sys.modules["torch._six"] = mock_six

print("✓ timm compatibility patch applied.")

### 1.6 — Verify Environment

In [ ]:
import torch, timm

print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")
else:
    print("⚠️  No GPU detected! Select GPU runtime: Runtime > Change runtime type > A100")
print(f"timm:       {timm.__version__}")

try:
    from mamba_ssm import Mamba
    print(f"mamba-ssm:  ✓ (real CUDA kernels)")
except ImportError as e:
    print(f"mamba-ssm:  ✗ FAILED - {e}")

# Add project to Python path
sys.path.insert(0, LOCAL_REPO)
print(f"\nProject root: {LOCAL_REPO}")

## Section 2: Checkpoint Verification

In [ ]:
!python -m tests.verify_checkpoint --ckpt_path "{CKPT_PATH}"

## Section 3: Smoke Tests

In [ ]:
print("=" * 55)
print("  Running smoke tests...")
print("=" * 55)

!python -m tests.test_teacher
!python -m tests.test_student
!python -m tests.test_loss
!python -m tests.test_augmentations
!python -m tests.test_end_to_end

print("\n" + "=" * 55)
print("  Smoke tests complete.")
print("=" * 55)


> ⛔ **If any test fails, STOP HERE.** Debug the failing component before proceeding to training.

## Section 4: Federated Dataset Inspection

### 4.1 — Verify Client Split Structure

In [ ]:
import os, sys
sys.path.insert(0, LOCAL_REPO)

# Check that the federated split CSVs exist
split_dir = os.path.join(DRIVE_DATASET, f"{N_CLIENTS}_clients", SPLIT_TYPE)
print(f"Looking for client splits in: {split_dir}")
print()

if os.path.isdir(split_dir):
    files = sorted(os.listdir(split_dir))
    total_samples = 0
    for f in files:
        fp = os.path.join(split_dir, f)
        if f.endswith('.csv'):
            with open(fp) as fh:
                n = len([l for l in fh if l.strip()])
            total_samples += n
            print(f"  ✓ {f}: {n} samples")
    print(f"\n  Total across all clients: {total_samples} samples")
else:
    print(f"  ✗ Directory not found: {split_dir}")
    print(f"  Run data/data_split.py to generate federated splits.")

### 4.2 — Client Data Distribution

Visualize the class distribution per client to assess non-IID degree.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

split_dir = os.path.join(DRIVE_DATASET, f"{N_CLIENTS}_clients", SPLIT_TYPE)
labels_path = os.path.join(DRIVE_DATASET, "labels.csv")

# Load global labels
labels = {}
with open(labels_path) as f:
    for line in f:
        parts = line.strip().split(",")
        if len(parts) >= 2:
            labels[parts[0]] = int(float(parts[1]))

# Count per-client class distribution
client_dists = {}
for cid in range(1, N_CLIENTS + 1):
    csv_path = os.path.join(split_dir, f"client_{cid}.csv")
    with open(csv_path) as f:
        fnames = [l.strip().split(",")[0] for l in f if l.strip()]
    class_counts = {}
    for fn in fnames:
        lbl = labels.get(fn, -1)
        class_counts[lbl] = class_counts.get(lbl, 0) + 1
    client_dists[cid] = class_counts

# Plot
all_classes = sorted(set(c for d in client_dists.values() for c in d))
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_CLIENTS)
width = 0.8 / len(all_classes)

for i, cls in enumerate(all_classes):
    counts = [client_dists[cid].get(cls, 0) for cid in range(1, N_CLIENTS + 1)]
    ax.bar(x + i * width, counts, width, label=f"Class {cls}")

ax.set_xlabel("Client", fontsize=12)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_title(f"Client Data Distribution ({SPLIT_TYPE})", fontsize=14, fontweight='bold')
ax.set_xticks(x + width * len(all_classes) / 2)
ax.set_xticklabels([f"Client {i}" for i in range(1, N_CLIENTS + 1)])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_client_distribution.png"), dpi=150)
plt.show()

### 4.3 — Visualize Augmentation Pairs (Client 1 Sample)

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from augmentations.medical_aug import DualViewDataset, get_teacher_transform, get_student_transform
from augmentations.retina_dataset import RetinaDataset

# The exact normalization stats used in medical_aug.py
RETINA_MEAN = [0.5007, 0.5010, 0.5019]
RETINA_STD = [0.0342, 0.0535, 0.0484]

def denormalize(tensor):
    """Reverses the Retina normalization so the image can be displayed properly."""
    mean = torch.tensor(RETINA_MEAN).view(3, 1, 1).to(tensor.device)
    std = torch.tensor(RETINA_STD).view(3, 1, 1).to(tensor.device)
    return (tensor * std + mean).clamp(0, 1)

# Load client 1 data
split_csv = os.path.join(f"{N_CLIENTS}_clients", SPLIT_TYPE, "client_1.csv")
base_ds = RetinaDataset(
    data_path=DRIVE_DATASET, phase="train",
    split_type="federated", split_csv=split_csv,
)
dual_ds = DualViewDataset(
    base_ds,
    teacher_transform=get_teacher_transform(dataset="retina"),
    student_transform=get_student_transform(dataset="retina"),
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Augmentation Pairs (Client 1, {SPLIT_TYPE})", fontsize=14, fontweight='bold')

for i in range(4):
    t_view, s_view = dual_ds[i * 10]

    # Apply proper denormalization before converting to PIL
    t_clean = TF.to_pil_image(denormalize(t_view))
    s_clean = TF.to_pil_image(denormalize(s_view))

    axes[0, i].imshow(t_clean)
    axes[0, i].set_title(f"Teacher #{i}", fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(s_clean)
    axes[1, i].set_title(f"Student #{i}", fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_augmentation_pairs.png"), dpi=150)
plt.show()


## Section 5: Federated Pre-training

This section launches `train_fedavg.py` which orchestrates the full federated loop:
1. For each communication round, each of the 5 clients trains locally for `E_epoch` epochs
2. Client models are aggregated via weighted averaging (FedAvg)
3. If `mu > 0`, a FedProx proximal penalty prevents client drift
4. Global model is broadcast back to all clients

**Resume support**: If the notebook disconnects, re-run from Section 1 through here — `train_fedavg.py` will auto-resume from `ckpt_latest.pth`.

In [ ]:
import os

# Construct the correct path using the dataset directory
corrupt_file = os.path.join(DRIVE_DATASET, "train", "11070_right_test.npy")

if os.path.exists(corrupt_file):
    os.remove(corrupt_file)
    print(f"✓ Removed corrupt file: {corrupt_file}")
else:
    print(f"File not found (already removed or incorrect path): {corrupt_file}")


### 5.1 — Launch Federated Training

In [ ]:
!python train_fedavg.py \
    --data_path "{DRIVE_DATASET}" \
    --teacher_ckpt "{CKPT_PATH}" \
    --output_dir "{DRIVE_OUTPUT}" \
    --split_type {SPLIT_TYPE} \
    --n_clients {N_CLIENTS} \
    --max_rounds {MAX_ROUNDS} \
    --E_epoch {E_EPOCH} \
    --mu {MU} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --num_workers {NUM_WORKERS} \
    --save_every 10 \
    --device cuda


### 5.2 — Backup Checkpoints to Google Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy checkpoints and metrics to Drive for persistence
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp) and (f.endswith('.pth') or f.endswith('.csv')):
        shutil.copy2(fp, os.path.join(DRIVE_OUTPUT, f))
        size = os.path.getsize(fp) / 1e6
        print(f"  ✓ {f} ({size:.1f} MB)")

print(f"\nCheckpoints backed up to: {DRIVE_OUTPUT}")
print("These persist even after Colab disconnects.")

## Section 6: Training Diagnostics & Visualization

The federated metrics CSV (`federated_metrics.csv`) contains per-round and per-client metrics.
This section visualises convergence, client alignment, and system health.

### 6.0 — Load Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

local_metrics = os.path.join(OUTPUT_DIR, "federated_metrics.csv")
drive_metrics = os.path.join(DRIVE_OUTPUT, "federated_metrics.csv")

if os.path.exists(local_metrics):
    METRICS_CSV = local_metrics
elif os.path.exists(drive_metrics):
    METRICS_CSV = drive_metrics
else:
    raise FileNotFoundError("federated_metrics.csv not found locally or on Drive.")

df = pd.read_csv(METRICS_CSV)
print(f"Loaded {len(df)} rounds from {METRICS_CSV}")
print(f"Columns: {list(df.columns)}")
df.tail()

### 6.1 — Global SALT Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['round'], df['avg_loss'], color='#2196F3', linewidth=2, label='Weighted Avg Loss')
ax.fill_between(df['round'], df['avg_loss'] * 0.9, df['avg_loss'] * 1.1,
                alpha=0.1, color='#2196F3')

ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target (< 0.3)')
ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('SALT Loss', fontsize=12)
ax.set_title(f'{ALGO_NAME} Global Loss ({SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_global_loss.png"), dpi=150)
plt.show()

print(f"Final loss: {df['avg_loss'].iloc[-1]:.4f}")
print(f"Best loss:  {df['avg_loss'].min():.4f} (round {df['avg_loss'].idxmin() + 1})")

### 6.2 — Per-Client Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
colors = ['#E91E63', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']

for i, col in enumerate(client_cols):
    color = colors[i % len(colors)]
    ax.plot(df['round'], df[col], linewidth=1.5, alpha=0.8,
            color=color, label=f'Client {i+1}')

# Overlay global average
ax.plot(df['round'], df['avg_loss'], color='black', linewidth=2.5,
        linestyle='--', label='Global Avg', zorder=10)

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('SALT Loss', fontsize=12)
ax.set_title(f'Per-Client Loss Curves ({ALGO_NAME}, {SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_per_client_loss.png"), dpi=150)
plt.show()

### 6.3 — Client Loss Variance (Convergence Alignment)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
client_df = df[client_cols]
loss_var = client_df.var(axis=1)
loss_std = client_df.std(axis=1)

ax.plot(df['round'], loss_var, color='#FF5722', linewidth=2, label='Variance')
ax.fill_between(df['round'], 0, loss_var, alpha=0.15, color='#FF5722')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Loss Variance Across Clients', fontsize=12)
ax.set_title(f'Client Convergence Alignment ({ALGO_NAME}, {SPLIT_TYPE})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_client_variance.png"), dpi=150)
plt.show()

print(f"Initial variance: {loss_var.iloc[0]:.6f}")
print(f"Final variance:   {loss_var.iloc[-1]:.6f}")
print(f"Reduction:        {(1 - loss_var.iloc[-1] / max(loss_var.iloc[0], 1e-10)) * 100:.1f}%")

### 6.4 — Embedding Std (Collapse Detection)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df['round'], df['avg_enc_std'], color='#E91E63', linewidth=2, label='Avg enc_std')
ax.axhline(y=0.1, color='#4CAF50', linestyle='--', alpha=0.7, label='Healthy (> 0.1)')
ax.axhline(y=0.02, color='#F44336', linestyle='--', alpha=0.7, label='Collapse (< 0.02)')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Embedding Std', fontsize=12)
ax.set_title(f'Encoder Embedding Std ({ALGO_NAME}, {SPLIT_TYPE})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_enc_std.png"), dpi=150)
plt.show()

final_std = df['avg_enc_std'].iloc[-1]
print(f"Final enc_std: {final_std:.4f}  {'✓ HEALTHY' if final_std > 0.1 else '✗ WARNING' if final_std > 0.02 else '✗✗ COLLAPSED'}")

### 6.5 — Per-Round Training Time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(df['round'], df['round_time_s'], color='#607D8B', alpha=0.7, width=1.0)
mean_time = df['round_time_s'].mean()
ax.axhline(y=mean_time, color='#FF5722', linestyle='--',
           label=f"Mean: {mean_time:.1f}s")

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Per-Round Training Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_round_timing.png"), dpi=150)
plt.show()

total_hours = df['round_time_s'].sum() / 3600
print(f"Total training time: {total_hours:.2f} hours")
print(f"Average per round:   {mean_time:.1f}s ({N_CLIENTS} clients x {E_EPOCH} local epochs)")

### 6.6 — GPU Memory Usage

In [ ]:
import torch

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df['round'], df['gpu_mb'] / 1024, color='#3F51B5', linewidth=2, label='Allocated')

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    ax.axhline(y=total_gb, color='gray', linestyle=':', alpha=0.5,
               label=f'Total VRAM ({total_gb:.0f} GB)')

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('GPU Memory (GB)', fontsize=12)
ax.set_title('GPU Memory Usage Over Training', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_gpu_memory.png"), dpi=150)
plt.show()

print(f"Peak GPU memory: {df['gpu_mb'].max():.0f} MB ({df['gpu_mb'].max() / 1024:.1f} GB)")

### 6.7 — Combined Federated Training Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'FedMamba-SALT {ALGO_NAME} Dashboard ({SPLIT_TYPE})', fontsize=16, fontweight='bold')

client_cols = [c for c in df.columns if c.startswith('client_') and c.endswith('_loss')]
colors = ['#E91E63', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']

# --- Global Loss ---
ax = axes[0, 0]
ax.plot(df['round'], df['avg_loss'], color='#2196F3', linewidth=2)
ax.axhline(y=0.3, color='#4CAF50', linestyle='--', alpha=0.7, label='Target')
ax.set_ylabel('Loss'); ax.set_title('Global SALT Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Per-Client Loss ---
ax = axes[0, 1]
for i, col in enumerate(client_cols):
    ax.plot(df['round'], df[col], linewidth=1.2, alpha=0.7, color=colors[i % len(colors)],
            label=f'C{i+1}')
ax.set_title('Per-Client Loss')
ax.legend(fontsize=7, ncol=3); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Client Variance ---
ax = axes[0, 2]
client_var = df[client_cols].var(axis=1)
ax.plot(df['round'], client_var, color='#FF5722', linewidth=2)
ax.set_title('Client Loss Variance'); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Embedding Std ---
ax = axes[1, 0]
ax.plot(df['round'], df['avg_enc_std'], color='#E91E63', linewidth=2)
ax.axhline(y=0.02, color='#F44336', linestyle='--', label='Collapse')
ax.set_ylabel('Std'); ax.set_title('Enc Std'); ax.set_xlabel('Round')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

# --- Round Time ---
ax = axes[1, 1]
ax.bar(df['round'], df['round_time_s'], color='#607D8B', alpha=0.7, width=1.0)
ax.set_title('Round Time (s)'); ax.set_xlabel('Round')
ax.grid(True, alpha=0.3, axis='y')

# --- GPU Memory ---
ax = axes[1, 2]
ax.plot(df['round'], df['gpu_mb'] / 1024, color='#3F51B5', linewidth=2)
ax.set_title('GPU Memory (GB)'); ax.set_xlabel('Round')
ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot_federated_dashboard.png"), dpi=150, bbox_inches='tight')
plt.show()

### 6.8 — Final Health Check

In [ ]:
import torch
from objectives.salt_loss import embedding_std
from models.inception_mamba import InceptionMambaEncoder

ckpt_path = os.path.join(OUTPUT_DIR, "ckpt_latest.pth")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(DRIVE_OUTPUT, "ckpt_latest.pth")

ckpt = torch.load(ckpt_path, map_location="cpu")
final_round = ckpt['comm_round'] + 1
final_loss = ckpt['loss']

# Load student and compute embedding std on random data
student = InceptionMambaEncoder(patch_size=16, embed_dim=448, depth=6, out_dim=768)
student.load_state_dict(ckpt['student_state_dict'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student.to(device)
student.eval()

with torch.no_grad():
    dummy = torch.randn(16, 3, 224, 224).to(device)
    emb = student(dummy)
    std_val = embedding_std(emb)

total_time = df['round_time_s'].sum() / 3600

print("=" * 55)
print(f"  {ALGO_NAME} Training Health Check ({SPLIT_TYPE})")
print("=" * 55)
print(f"  Final round:     {final_round}")
print(f"  Final loss:      {final_loss:.4f}  {'✓ PASS' if final_loss < 0.3 else '✗ FAIL'} (target < 0.3)")
print(f"  Embedding std:   {std_val:.4f}  {'✓ PASS' if std_val > 0.05 else '✗ FAIL'} (target > 0.05)")
print(f"  Peak GPU memory: {df['gpu_mb'].max():.0f} MB ({df['gpu_mb'].max() / 1024:.1f} GB)")
print(f"  Total time:      {total_time:.2f} hours")
print("=" * 55)

## Section 7: Linear Probe Evaluation

Freeze the federated pre-trained encoder, attach a single linear layer, train only that layer on labeled data.

### 7.1 — Run Linear Probe

In [ ]:
EVAL_LP_DIR = os.path.join(OUTPUT_DIR, "eval_linear_probe")

local_ckpt_path = os.path.join(OUTPUT_DIR, "ckpt_latest.pth")
drive_ckpt_path = os.path.join(DRIVE_OUTPUT, "ckpt_latest.pth")

if os.path.exists(local_ckpt_path):
    FINAL_CKPT = local_ckpt_path
    print(f"Using local checkpoint: {FINAL_CKPT}")
elif os.path.exists(drive_ckpt_path):
    FINAL_CKPT = drive_ckpt_path
    print(f"Using Drive checkpoint: {FINAL_CKPT}")
else:
    raise FileNotFoundError(f"Checkpoint not found locally ({local_ckpt_path}) or on Drive ({drive_ckpt_path})")

In [ ]:
!python -m eval.linear_probe \
    --encoder_ckpt "{FINAL_CKPT}" \
    --data_path "{DRIVE_DATASET}" \
    --num_classes {NUM_CLASSES} \
    --output_dir "{EVAL_LP_DIR}" \
    --epochs 100 \
    --batch_size 128 \
    --lr 1.0e-3\
    --mode linear_probe

### 7.2 — Display Confusion Matrix

In [ ]:
from IPython.display import Image as IPImage, display
import glob

cm_files = glob.glob(os.path.join(EVAL_LP_DIR, "*confusion*")) + \
           glob.glob(os.path.join(EVAL_LP_DIR, "**/*confusion*"), recursive=True)
if cm_files:
    for f in cm_files:
        print(f"Confusion matrix: {f}")
        display(IPImage(filename=f))
else:
    print("No confusion matrix found. Check eval output directory.")

## Section 8: Federated Fine-tuning

Fine-tune the pre-trained federated encoder in a fully federated supervised stage. Each client trains locally on its own labeled split, then the global encoder and classifier are aggregated each round.


In [ ]:
# ============================================================
# FEDERATED FINE-TUNING SETTINGS
# ============================================================
FED_FT_MODE        = "federated_finetune"
FED_FT_ALGO        = "fedavgm"              # server momentum aggregation
FED_FT_MAX_ROUNDS  = 80
FED_FT_E_EPOCH     = 3
FED_FT_LR          = 1e-3
FED_FT_BATCH_SIZE  = 64
FED_FT_MU          = 0.01                 # >0 keeps FedProx active
FED_FT_USE_MIXUP   = False
FED_FT_FOCAL       = False

FED_FT_ALGO_NAME = "FedProx+ServerMomentum" if FED_FT_ALGO == "fedavgm" and FED_FT_MU > 0 else FED_FT_ALGO

# Use the federated SSL pretraining checkpoint, not a fine-tuned checkpoint.
pretrain_candidates = [
    os.path.join(OUTPUT_DIR, "ckpt_best.pth"),
    os.path.join(DRIVE_OUTPUT, "ckpt_best.pth"),
    os.path.join(OUTPUT_DIR, "ckpt_latest.pth"),
    os.path.join(DRIVE_OUTPUT, "ckpt_latest.pth"),
]
FINAL_CKPT = next((p for p in pretrain_candidates if os.path.exists(p)), None)
if FINAL_CKPT is None:
    raise FileNotFoundError("No federated pretraining checkpoint found. Run Section 5 first.")

EVAL_FED_FT_DIR = os.path.join(OUTPUT_DIR, f"eval_{FED_FT_MODE}")
os.makedirs(EVAL_FED_FT_DIR, exist_ok=True)

print(f"Mode:          {FED_FT_MODE}")
print(f"Algorithm:     {FED_FT_ALGO_NAME}  (--algo {FED_FT_ALGO}, mu={FED_FT_MU})")
print(f"Rounds:        {FED_FT_MAX_ROUNDS}  |  E_epoch: {FED_FT_E_EPOCH}")
print(f"LR:            {FED_FT_LR}  |  Batch: {FED_FT_BATCH_SIZE}")
print(f"Output dir:    {EVAL_FED_FT_DIR}")
print(f"Pretrain ckpt: {FINAL_CKPT}")


### 8.1 - Launch Federated Fine-tuning

In [ ]:
mixup_flag = "--use_mixup" if FED_FT_USE_MIXUP else ""
focal_flag = "--use_focal_loss" if FED_FT_FOCAL else ""

cmd = (
    f"python train_fed_finetune.py"
    f' --encoder_ckpt "{FINAL_CKPT}"'
    f' --data_path "{DRIVE_DATASET}"'
    f" --num_classes {NUM_CLASSES}"
    f' --output_dir "{EVAL_FED_FT_DIR}"'
    f" --n_clients {N_CLIENTS}"
    f" --split_type {SPLIT_TYPE}"
    f" --max_rounds {FED_FT_MAX_ROUNDS}"
    f" --E_epoch {FED_FT_E_EPOCH}"
    f" --lr {FED_FT_LR}"
    f" --batch_size {FED_FT_BATCH_SIZE}"
    f" --mode {FED_FT_MODE}"
    f" --algo {FED_FT_ALGO}"
    f" --mu {FED_FT_MU}"
    f" {mixup_flag} {focal_flag}"
)

print(cmd)
!{cmd}


### 8.2 - Backup Federated Fine-tune Results to Drive

In [ ]:
import shutil

DRIVE_FED_FT_OUTPUT = os.path.join(DRIVE_OUTPUT, f"eval_{FED_FT_MODE}")
os.makedirs(DRIVE_FED_FT_OUTPUT, exist_ok=True)

for f in sorted(os.listdir(EVAL_FED_FT_DIR)):
    fp = os.path.join(EVAL_FED_FT_DIR, f)
    if os.path.isfile(fp):
        shutil.copy2(fp, os.path.join(DRIVE_FED_FT_OUTPUT, f))
        print(f"  Saved {f}")

print(f"Backed up to: {DRIVE_FED_FT_OUTPUT}")


### 8.3 - Per-Round Metrics & Diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

FED_FT_CSV = os.path.join(EVAL_FED_FT_DIR, "fed_finetune_metrics.csv")
if not os.path.exists(FED_FT_CSV):
    FED_FT_CSV = os.path.join(DRIVE_OUTPUT, f"eval_{FED_FT_MODE}", "fed_finetune_metrics.csv")

if os.path.exists(FED_FT_CSV):
    df_ft = pd.read_csv(FED_FT_CSV)
    print(f"Loaded {len(df_ft)} rounds from {FED_FT_CSV}")
    print(df_ft[["round", "val_acc", "auc", "enc_lr", "cls_lr"]].tail(5).to_string(index=False))
else:
    df_ft = None
    print(f"Metrics CSV not found: {FED_FT_CSV}")


In [ ]:
# Per-round validation accuracy, AUC, and LR schedule
if df_ft is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"Federated Fine-tune ({FED_FT_MODE}, {SPLIT_TYPE}, {ALGO_NAME})",
        fontsize=14, fontweight="bold"
    )

    ax = axes[0]
    ax.plot(df_ft["round"], df_ft["val_acc"], color="#2ecc71", linewidth=2)
    best_idx = df_ft["val_acc"].idxmax()
    ax.axvline(x=df_ft["round"].iloc[best_idx], color="#27ae60", linestyle="--", alpha=0.6)
    ax.annotate(
        f"Best: {df_ft['val_acc'].iloc[best_idx]:.1f}%",
        xy=(df_ft["round"].iloc[best_idx], df_ft["val_acc"].iloc[best_idx]),
        fontsize=9, fontweight="bold", color="#27ae60",
    )
    ax.set_xlabel("Round"); ax.set_ylabel("Validation Accuracy (%)")
    ax.set_title("Global Val Accuracy"); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(df_ft["round"], df_ft["auc"], color="#8e44ad", linewidth=2)
    ax.set_xlabel("Round"); ax.set_ylabel("AUC")
    ax.set_title("Global Validation AUC"); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(df_ft["round"], df_ft["enc_lr"], label="Encoder LR", color="#9b59b6", linewidth=2)
    ax.plot(df_ft["round"], df_ft["cls_lr"], label="Classifier LR", color="#f39c12", linewidth=2)
    ax.set_xlabel("Round"); ax.set_ylabel("Learning Rate")
    ax.set_title("LR Schedule"); ax.set_yscale("log"); ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(EVAL_FED_FT_DIR, "plot_fed_ft_curves.png")
    plt.savefig(plot_path, dpi=150)
    plt.show()

    print(f"Best val acc: {df_ft['val_acc'].max():.2f}% at round {df_ft.loc[df_ft['val_acc'].idxmax(), 'round']}")
    print(f"Final AUC:    {df_ft['auc'].iloc[-1]:.4f}")
else:
    print("No metrics to plot.")


In [ ]:
# Per-client supervised loss convergence across rounds
if df_ft is not None:
    client_cols = [c for c in df_ft.columns if c.startswith("client_") and c.endswith("_loss")]

    if client_cols:
        colors = ["#E91E63", "#FF9800", "#4CAF50", "#2196F3", "#9C27B0"]
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle(f"Client Loss Diagnostics ({ALGO_NAME}, {SPLIT_TYPE})", fontsize=13, fontweight="bold")

        ax = axes[0]
        for i, col in enumerate(client_cols):
            ax.plot(df_ft["round"], df_ft[col], linewidth=1.5, alpha=0.8,
                    color=colors[i % len(colors)], label=f"C{i+1}")
        ax.set_xlabel("Round"); ax.set_ylabel("Loss")
        ax.set_title("Per-Client CE Loss")
        ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

        ax = axes[1]
        loss_var = df_ft[client_cols].var(axis=1)
        ax.plot(df_ft["round"], loss_var, color="#FF5722", linewidth=2)
        ax.fill_between(df_ft["round"], 0, loss_var, alpha=0.12, color="#FF5722")
        ax.set_xlabel("Round"); ax.set_ylabel("Loss Variance")
        ax.set_title("Client Convergence Alignment"); ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

        plt.tight_layout()
        plt.savefig(os.path.join(EVAL_FED_FT_DIR, "plot_fed_ft_client_loss.png"), dpi=150)
        plt.show()

        print(f"Initial variance: {loss_var.iloc[0]:.6f}")
        print(f"Final variance:   {loss_var.iloc[-1]:.6f}")
        print(f"Reduction:        {(1 - loss_var.iloc[-1] / max(loss_var.iloc[0], 1e-10)) * 100:.1f}%")
    else:
        print("No per-client loss columns found.")


### 8.4 - Confusion Matrix & ROC Curve

In [ ]:
from IPython.display import Image as IPImage, display
import glob

for pattern in ["*confusion*", "*roc_curve*", "*round_curves*"]:
    files = (glob.glob(os.path.join(EVAL_FED_FT_DIR, pattern)) +
             glob.glob(os.path.join(EVAL_FED_FT_DIR, "**", pattern), recursive=True))
    for f in sorted(files):
        print(f)
        display(IPImage(filename=f))


## Section 9: Test-Time Augmentation Evaluation

Run TTA on the best federated fine-tuned checkpoint. This produces a TTA summary CSV, comparison plot, and threshold-sweep plot under `eval_tta/`, then mirrors them to Drive.


In [ ]:
import os
import shutil

TTA_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "eval_tta")
os.makedirs(TTA_OUTPUT_DIR, exist_ok=True)

fed_ft_mode = globals().get("FED_FT_MODE", "federated_finetune")
fed_ft_local_dir = globals().get("EVAL_FED_FT_DIR", os.path.join(OUTPUT_DIR, f"eval_{fed_ft_mode}"))
fed_ft_drive_dir = os.path.join(DRIVE_OUTPUT, f"eval_{fed_ft_mode}")

TTA_CKPT_CANDIDATES = [
    os.path.join(fed_ft_local_dir, "ckpt_best_finetune.pth"),
    os.path.join(fed_ft_local_dir, "ckpt_latest.pth"),
    os.path.join(fed_ft_drive_dir, "ckpt_best_finetune.pth"),
    os.path.join(fed_ft_drive_dir, "ckpt_latest.pth"),
]
TTA_CKPT = next((p for p in TTA_CKPT_CANDIDATES if os.path.exists(p)), None)
if TTA_CKPT is None:
    raise FileNotFoundError(
        "No federated fine-tune checkpoint found for TTA. Run Section 8 first or verify the Drive backup path."
    )

print(f"TTA checkpoint: {TTA_CKPT}")
print(f"TTA output:     {TTA_OUTPUT_DIR}")

tta_cmd = (
    f"python -m eval.eval_tta"
    f' --ckpt "{TTA_CKPT}"'
    f' --data_path "{DRIVE_DATASET}"'
    f" --num_classes {NUM_CLASSES}"
    f' --split_type "{SPLIT_TYPE}"'
    f" --split_csv test.csv"
    f" --batch_size 64"
    f" --n_tta 8"
    f" --threshold_sweep"
    f' --output_dir "{TTA_OUTPUT_DIR}"'
    f" --device cuda"
)

print(tta_cmd)
!{tta_cmd}


In [ ]:
from IPython.display import Image as IPImage, display
import glob
import os
import shutil

DRIVE_TTA_OUTPUT = os.path.join(DRIVE_OUTPUT, "eval_tta")
if os.path.exists(DRIVE_TTA_OUTPUT):
    shutil.rmtree(DRIVE_TTA_OUTPUT)
shutil.copytree(TTA_OUTPUT_DIR, DRIVE_TTA_OUTPUT)
print(f"TTA artifacts backed up to: {DRIVE_TTA_OUTPUT}")

for pattern in ["plot_tta_comparison.png", "plot_threshold_sweep.png"]:
    files = glob.glob(os.path.join(TTA_OUTPUT_DIR, pattern))
    for f in files:
        print(f)
        display(IPImage(filename=f))


## Section 10: Results Summary & Drive Backup


### 10.1 - Final Results


In [ ]:
import os
import csv
import shutil
import pandas as pd
import matplotlib.pyplot as plt

if 'df' not in locals() and 'df' not in globals():
    METRICS_CSV = os.path.join(OUTPUT_DIR, "federated_metrics.csv")
    if not os.path.exists(METRICS_CSV):
        METRICS_CSV = os.path.join(DRIVE_OUTPUT, "federated_metrics.csv")
    df = pd.read_csv(METRICS_CSV) if os.path.exists(METRICS_CSV) else None

if df is not None:
    final_loss = df['avg_loss'].iloc[-1]
    best_loss = df['avg_loss'].min()
    best_round = int(df.loc[df['avg_loss'].idxmin(), 'round'])
    std_val = df['avg_enc_std'].iloc[-1]
    total_time = df['round_time_s'].sum() / 3600
else:
    final_loss = best_loss = 0.0
    best_round = 0
    std_val = 0.0
    total_time = 0.0

if 'EVAL_LP_DIR' not in locals():
    EVAL_LP_DIR = os.path.join(OUTPUT_DIR, "eval_linear_probe")
if 'FED_FT_MODE' not in locals():
    FED_FT_MODE = "federated_finetune"
if 'EVAL_FED_FT_DIR' not in locals():
    EVAL_FED_FT_DIR = os.path.join(OUTPUT_DIR, f"eval_{FED_FT_MODE}")
if 'TTA_OUTPUT_DIR' not in locals():
    TTA_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "eval_tta")

def get_report_accuracy(eval_dir):
    if os.path.exists(eval_dir):
        for root, dirs, files in os.walk(eval_dir):
            for file in files:
                if file.startswith("classification_report") and file.endswith(".txt"):
                    with open(os.path.join(root, file)) as f:
                        for line in f:
                            if 'weighted avg' in line:
                                return float(line.split()[3]) * 100
    return None

def get_tta_metrics(eval_dir):
    csv_path = os.path.join(eval_dir, "tta_summary.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(DRIVE_OUTPUT, "eval_tta", "tta_summary.csv")
    if not os.path.exists(csv_path):
        return None, None
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("setting") == "tta":
                return float(row["accuracy"]), float(row["auc"])
    return None, None

lp_acc = get_report_accuracy(EVAL_LP_DIR)
fed_ft_acc = get_report_accuracy(EVAL_FED_FT_DIR)
tta_acc, tta_auc = get_tta_metrics(TTA_OUTPUT_DIR)
baseline_acc = 81.93
centralized_ref_acc = 82.23

ok_loss = "OK" if final_loss < 0.3 else "WARN"
ok_std = "OK" if std_val > 0.05 else "WARN"

print("=" * 68)
print(f"  FedMamba-SALT: {ALGO_NAME} Split 1 Results")
print("=" * 68)
print(f"  Pretraining:        {ALGO_NAME} (mu={MU}, E_epoch={E_EPOCH}, rounds={MAX_ROUNDS})")
print(f"  Final SALT loss:    {final_loss:.4f}  [{ok_loss}]")
print(f"  Best SALT loss:     {best_loss:.4f}  (round {best_round})")
print(f"  Embedding std:      {std_val:.4f}  [{ok_std}]")
print(f"  Pretrain time:      {total_time:.2f} hours")
print("-" * 68)
print(f"  Linear Probe:       {lp_acc:.2f}%" if lp_acc is not None else "  Linear Probe:       not found")
print(f"  Federated FT:       {fed_ft_acc:.2f}%" if fed_ft_acc is not None else "  Federated FT:       not found")
if tta_acc is not None:
    print(f"  Federated FT + TTA: {tta_acc:.2f}%  (AUC={tta_auc:.4f})")
else:
    print("  Federated FT + TTA: not found")
print(f"  Centralized ref:    {centralized_ref_acc:.2f}%")
print(f"  Fed-MAE baseline:   {baseline_acc:.2f}%")
print("=" * 68)

models = []
accuracies = []
colors = []

if lp_acc is not None:
    models.append(f'{ALGO_NAME}\nLinear Probe')
    accuracies.append(lp_acc)
    colors.append('#FFC107')
if fed_ft_acc is not None:
    models.append('Federated\nFine-tune')
    accuracies.append(fed_ft_acc)
    colors.append('#2ECC71')
if tta_acc is not None:
    models.append('Federated FT\n+ TTA')
    accuracies.append(tta_acc)
    colors.append('#2196F3')

models.extend(['Centralized\nReference', 'Fed-MAE\nBaseline'])
accuracies.extend([centralized_ref_acc, baseline_acc])
colors.extend(['#8BC34A', '#E91E63'])

fig_width = max(9, 1.7 * len(models))
fig, ax = plt.subplots(figsize=(fig_width, 5.5))
bars = ax.bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.4, width=0.62)
patterns = ['//', 'xx', '\\\\', '..', '--']
for bar, pattern in zip(bars, patterns):
    bar.set_hatch(pattern)

ax.set_ylim(max(0, min(accuracies) - 10), max(accuracies) + 5)
ax.set_ylabel('Top-1 Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title(f'FedMamba-SALT Split 1 Evaluation', fontsize=14, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(baseline_acc, color='#E91E63', linestyle='--', linewidth=1.4, zorder=0, alpha=0.65)

plt.tight_layout()
summary_plot_path = os.path.join(OUTPUT_DIR, 'plot_final_summary.png')
plt.savefig(summary_plot_path, dpi=150)
if os.path.exists(DRIVE_OUTPUT):
    shutil.copy2(summary_plot_path, os.path.join(DRIVE_OUTPUT, 'plot_final_summary.png'))
plt.show()


### 10.2 - Paper Artifact Checklist & Drive Upload

This cell mirrors pretraining, linear-probe, federated fine-tuning, and TTA artifacts to Drive, gathers paper figures into one folder, writes a manifest, and verifies that all required paper plots exist.


In [ ]:
import os
import shutil
import csv
from pathlib import Path

output_root = Path(OUTPUT_DIR)
drive_root = Path(DRIVE_OUTPUT)
if not output_root.exists():
    raise FileNotFoundError(f"Local output directory not found: {output_root}")

drive_root.mkdir(parents=True, exist_ok=True)
RESULTS_BACKUP_DIR = drive_root / "all_results"
PAPER_ARTIFACT_DIR = drive_root / "paper_artifacts"
PAPER_FIG_DIR = PAPER_ARTIFACT_DIR / "figures"
PAPER_REPORT_DIR = PAPER_ARTIFACT_DIR / "tables_reports"

for path in [RESULTS_BACKUP_DIR, PAPER_ARTIFACT_DIR]:
    if path.exists():
        shutil.rmtree(path)

RESULTS_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_REPORT_DIR.mkdir(parents=True, exist_ok=True)

if 'FED_FT_MODE' not in locals():
    FED_FT_MODE = "federated_finetune"
fed_ft_artifact_dir = f"eval_{FED_FT_MODE}"

source_roots = [("local", output_root)]
if drive_root.resolve() != output_root.resolve():
    source_roots.append(("drive", drive_root))

required_plot_groups = {
    "client_distribution": ["plot_client_distribution.png"],
    "augmentation_pairs": ["plot_augmentation_pairs.png"],
    "global_ssl_loss": ["plot_global_loss.png"],
    "per_client_ssl_loss": ["plot_per_client_loss.png"],
    "client_loss_variance": ["plot_client_variance.png"],
    "embedding_std": ["plot_enc_std.png"],
    "round_timing": ["plot_round_timing.png"],
    "gpu_memory": ["plot_gpu_memory.png"],
    "federated_training_dashboard": ["plot_federated_dashboard.png"],
    "linear_probe_confusion": ["eval_linear_probe/**/confusion_matrix_linear_probe*.png"],
    "federated_finetune_curves": [f"{fed_ft_artifact_dir}/**/round_curves_*.png", f"{fed_ft_artifact_dir}/plot_fed_ft_curves.png"],
    "federated_finetune_client_loss": [f"{fed_ft_artifact_dir}/plot_fed_ft_client_loss.png"],
    "federated_finetune_roc": [f"{fed_ft_artifact_dir}/**/roc_curve_*.png"],
    "federated_finetune_confusion": [f"{fed_ft_artifact_dir}/**/confusion_matrix_*.png"],
    "tta_comparison": ["eval_tta/plot_tta_comparison.png"],
    "tta_threshold_sweep": ["eval_tta/plot_threshold_sweep.png"],
    "final_result_summary": ["plot_final_summary.png"],
}

paper_dirs = {"eval_linear_probe", fed_ft_artifact_dir, "eval_tta"}
core_top_level_plots = {
    pattern
    for patterns in required_plot_groups.values()
    for pattern in patterns
    if "/" not in pattern and "*" not in pattern
}

skip_roots = [RESULTS_BACKUP_DIR, PAPER_ARTIFACT_DIR]

def is_under(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

def is_paper_artifact(rel: Path) -> bool:
    suffix = rel.suffix.lower()
    allowed_suffixes = {".png", ".pdf", ".svg", ".csv", ".txt", ".json", ".md", ".pth"}
    if rel.parts and rel.parts[0] in paper_dirs:
        return suffix in allowed_suffixes
    if len(rel.parts) != 1:
        return False
    name = rel.name
    if name in core_top_level_plots:
        return True
    if name == "federated_metrics.csv":
        return True
    if name.startswith("ckpt_") and suffix == ".pth":
        return True
    return False

def iter_paper_files():
    seen = set()
    for root_label, root in source_roots:
        if not root.exists():
            continue
        for src in root.rglob("*"):
            if not src.is_file():
                continue
            if any(is_under(src, skip_root) for skip_root in skip_roots):
                continue
            rel = src.relative_to(root)
            if not is_paper_artifact(rel):
                continue
            key = str(src.resolve())
            if key in seen:
                continue
            seen.add(key)
            yield root_label, root, src

mirror_count = 0
for root_label, root, src in iter_paper_files():
    rel = Path(root_label) / src.relative_to(root)
    dst = RESULTS_BACKUP_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    mirror_count += 1

for src in output_root.iterdir():
    if src.is_file() and is_paper_artifact(src.relative_to(output_root)):
        shutil.copy2(src, drive_root / src.name)

def flat_name(root_label: str, root: Path, path: Path) -> str:
    return root_label + "__" + "__".join(path.relative_to(root).parts)

figure_suffixes = {".png", ".pdf", ".svg"}
report_suffixes = {".csv", ".txt", ".json", ".md"}
manifest_rows = []

for root_label, root, src in sorted(iter_paper_files(), key=lambda item: (item[0], str(item[2]))):
    suffix = src.suffix.lower()
    if suffix in figure_suffixes:
        dst = PAPER_FIG_DIR / flat_name(root_label, root, src)
        shutil.copy2(src, dst)
        manifest_rows.append(["figure", f"{root_label}:{src.relative_to(root)}", str(dst), src.stat().st_size])
    elif suffix in report_suffixes:
        dst = PAPER_REPORT_DIR / flat_name(root_label, root, src)
        shutil.copy2(src, dst)
        manifest_rows.append(["table_report", f"{root_label}:{src.relative_to(root)}", str(dst), src.stat().st_size])

def find_matches(patterns):
    matches = []
    for root_label, root in source_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.glob(pattern):
                if path.is_file() and is_paper_artifact(path.relative_to(root)):
                    matches.append((root_label, root, path))
    return sorted(matches, key=lambda item: (item[0], str(item[2])))

check_rows = []
missing = []
for group, patterns in required_plot_groups.items():
    matches = find_matches(patterns)
    rel_matches = [f"{root_label}:{path.relative_to(root)}" for root_label, root, path in matches]
    status = "OK" if matches else "MISSING"
    check_rows.append([group, status, "; ".join(rel_matches)])
    if not matches:
        missing.append(group)

manifest_path = PAPER_ARTIFACT_DIR / "paper_artifact_manifest.csv"
with open(manifest_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["artifact_type", "source_relative_path", "drive_artifact_path", "bytes"])
    writer.writerows(manifest_rows)

checklist_path = PAPER_ARTIFACT_DIR / "required_plot_checklist.csv"
with open(checklist_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["plot_group", "status", "matched_files"])
    writer.writerows(check_rows)

print("=" * 70)
print("Paper artifact upload complete")
print("=" * 70)
print(f"Source roots:         {', '.join(f'{label}={root}' for label, root in source_roots)}")
print(f"Drive output:         {drive_root}")
print(f"Output mirror:        {RESULTS_BACKUP_DIR} ({mirror_count} files)")
print(f"Paper figures:        {PAPER_FIG_DIR} ({sum(1 for _ in PAPER_FIG_DIR.glob('*'))} files)")
print(f"Tables/reports:       {PAPER_REPORT_DIR} ({sum(1 for _ in PAPER_REPORT_DIR.glob('*'))} files)")
print(f"Manifest:             {manifest_path}")
print(f"Required checklist:   {checklist_path}")

if missing:
    print("\nMissing required paper plot groups:")
    for group in missing:
        print(f"  - {group}")
    raise FileNotFoundError(
        "Paper artifact checklist is incomplete. Run the corresponding notebook sections, "
        "then rerun Section 10.2."
    )

print("\nAll required paper plot groups are present and uploaded to Drive.")


---

### Expected `federated_metrics.csv` columns

| Column | Description |
|---|---|
| `round` | Communication round (1-indexed) |
| `avg_loss` | Weighted average SALT loss across all clients |
| `avg_enc_std` | Weighted average encoder embedding std |
| `lr` | Current learning rate |
| `round_time_s` | Wall-clock time for the round (seconds) |
| `gpu_mb` | GPU memory allocated (MB) |
| `client_N_loss` | Per-client SALT loss (one column per client) |